In [1]:
import os
import random
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as T
from torchvision.models import vgg16

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ===== PATHS =====
TRAIN_LR = "/kaggle/input/competitions/dlp-jan-2026-nppe-3/train/train/input"
TRAIN_HR = "/kaggle/input/competitions/dlp-jan-2026-nppe-3/train/train/ground_truth"
TEST_DIR = "/kaggle/input/competitions/dlp-jan-2026-nppe-3/test/test/input"

OUTPUT_DIR = "/kaggle/working/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [2]:
class SRDataset(Dataset):
    def __init__(self, lr_dir, hr_dir, patch_size=96):
        self.lr_files = sorted(os.listdir(lr_dir))
        self.hr_files = sorted(os.listdir(hr_dir))
        self.lr_dir = lr_dir
        self.hr_dir = hr_dir
        self.ps = patch_size

        self.transform = T.Compose([
            T.ToTensor(),
            T.Normalize([0.5]*3, [0.5]*3)
        ])

    def __len__(self):
        return len(self.lr_files)

    def __getitem__(self, idx):
        lr = Image.open(os.path.join(self.lr_dir, self.lr_files[idx])).convert('RGB')
        hr = Image.open(os.path.join(self.hr_dir, self.hr_files[idx])).convert('RGB')

        w, h = lr.size
        x = random.randint(0, w - self.ps)
        y = random.randint(0, h - self.ps)

        lr = lr.crop((x, y, x+self.ps, y+self.ps))
        hr = hr.crop((x*4, y*4, (x+self.ps)*4, (y+self.ps)*4))

        return self.transform(lr), self.transform(hr)

In [3]:
# ===== EDSR MODEL =====
class ResBlock(nn.Module):
    def __init__(self, n_feats=64, res_scale=0.1):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(n_feats, n_feats, 3, 1, 1),
            nn.ReLU(True),
            nn.Conv2d(n_feats, n_feats, 3, 1, 1)
        )
        self.res_scale = res_scale

    def forward(self, x):
        return x + self.body(x) * self.res_scale

class EDSR(nn.Module):
    def __init__(self, n_resblocks=32, n_feats=64, scale=4):
        super().__init__()

        self.head = nn.Conv2d(3, n_feats, 3, 1, 1)

        self.body = nn.Sequential(
            *[ResBlock(n_feats) for _ in range(n_resblocks)]
        )

        self.conv_body = nn.Conv2d(n_feats, n_feats, 3, 1, 1)

        # Upsample
        modules = []
        for _ in range(2):  # x4
            modules += [
                nn.Conv2d(n_feats, n_feats * 4, 3, 1, 1),
                nn.PixelShuffle(2),
                nn.ReLU(True)
            ]
        self.upsample = nn.Sequential(*modules)

        self.tail = nn.Conv2d(n_feats, 3, 3, 1, 1)

    def forward(self, x):
        x = self.head(x)
        res = self.body(x)
        res = self.conv_body(res)
        x = x + res
        x = self.upsample(x)
        return self.tail(x)

In [4]:
# ===== DATA =====
dataset = SRDataset(TRAIN_LR, TRAIN_HR)
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size

train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=1)

# ===== MODEL =====
model = EDSR().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

l1 = nn.L1Loss()
l2 = nn.MSELoss()

# ===== PERCEPTUAL LOSS =====
vgg = vgg16(weights="IMAGENET1K_V1").features[:16].to(device).eval()
for p in vgg.parameters():
    p.requires_grad = False

def perceptual_loss(pred, target):
    return nn.functional.l1_loss(vgg(pred), vgg(target))

def calc_psnr(pred, target):
    mse = nn.functional.mse_loss(pred, target)
    return 10 * torch.log10(1 / mse)

# ===== AMP =====
scaler = torch.cuda.amp.GradScaler()

# ===== TRAIN =====
EPOCHS = 50

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for lr_img, hr_img in train_loader:
        lr_img, hr_img = lr_img.to(device), hr_img.to(device)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            pred = model(lr_img)

            loss = (
                0.8 * l1(pred, hr_img) +
                0.1 * l2(pred, hr_img) +
                0.1 * perceptual_loss(pred, hr_img)
            )

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    scheduler.step()

    # ===== VALIDATION =====
    model.eval()
    psnr_total = 0

    with torch.no_grad():
        for lr_img, hr_img in val_loader:
            lr_img, hr_img = lr_img.to(device), hr_img.to(device)
            pred = model(lr_img)
            psnr_total += calc_psnr(pred, hr_img).item()

    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f} | PSNR: {psnr_total/len(val_loader):.2f}")

torch.save(model.state_dict(), "/kaggle/working/model.pth")

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:02<00:00, 188MB/s]
/tmp/ipykernel_23/3302984971.py:33: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipykernel_23/3302984971.py:47: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 1 | Loss: 0.3507 | PSNR: 8.25
Epoch 2 | Loss: 0.3322 | PSNR: 8.88
Epoch 3 | Loss: 0.3106 | PSNR: 8.59
Epoch 4 | Loss: 0.2976 | PSNR: 10.20
Epoch 5 | Loss: 0.2975 | PSNR: 10.08
Epoch 6 | Loss: 0.2800 | PSNR: 10.70
Epoch 7 | Loss: 0.2766 | PSNR: 10.86
Epoch 8 | Loss: 0.2707 | PSNR: 10.97
Epoch 9 | Loss: 0.2644 | PSNR: 11.02
Epoch 10 | Loss: 0.2613 | PSNR: 11.45
Epoch 11 | Loss: 0.2634 | PSNR: 10.42
Epoch 12 | Loss: 0.2641 | PSNR: 10.60
Epoch 13 | Loss: 0.2676 | PSNR: 10.58
Epoch 14 | Loss: 0.2644 | PSNR: 11.14
Epoch 15 | Loss: 0.2619 | PSNR: 11.15
Epoch 16 | Loss: 0.2588 | PSNR: 10.67
Epoch 17 | Loss: 0.2615 | PSNR: 11.50
Epoch 18 | Loss: 0.2564 | PSNR: 11.55
Epoch 19 | Loss: 0.2544 | PSNR: 11.63
Epoch 20 | Loss: 0.2537 | PSNR: 11.37
Epoch 21 | Loss: 0.2607 | PSNR: 11.30
Epoch 22 | Loss: 0.2515 | PSNR: 11.55
Epoch 23 | Loss: 0.2497 | PSNR: 11.20
Epoch 24 | Loss: 0.2492 | PSNR: 11.67
Epoch 25 | Loss: 0.2541 | PSNR: 11.50
Epoch 26 | Loss: 0.2473 | PSNR: 11.72
Epoch 27 | Loss: 0.2472 

In [5]:
# ===== INFERENCE =====
def augment(x, mode):
    if mode == 0: return x
    if mode == 1: return x.flip(-1)
    if mode == 2: return x.flip(-2)
    if mode == 3: return x.transpose(-1, -2)
    if mode == 4: return x.flip(-1).transpose(-1, -2)
    if mode == 5: return x.flip(-2).transpose(-1, -2)
    if mode == 6: return x.flip(-1).flip(-2)
    if mode == 7: return x.flip(-1).flip(-2).transpose(-1, -2)

def deaugment(x, mode):
    return augment(x, mode)  # symmetric

# ===== INFERENCE =====
model.eval()

transform = T.Compose([
    T.ToTensor(),
    T.Normalize([0.5]*3, [0.5]*3)
])

to_pil = T.ToPILImage()

for file in sorted(os.listdir(TEST_DIR)):
    img = Image.open(os.path.join(TEST_DIR, file)).convert('RGB')
    inp = transform(img).unsqueeze(0).to(device)

    preds = []
    with torch.no_grad():
        for m in range(8):  # 🔥 self-ensemble
            aug = augment(inp, m)
            out = model(aug)
            out = deaugment(out, m)
            preds.append(out)

    out = torch.stack(preds).mean(0)

    out = (out * 0.5 + 0.5).clamp(0,1)
    out_img = to_pil(out.squeeze().cpu())
    out_img.save(os.path.join(OUTPUT_DIR, file))

In [6]:
import os
import pandas as pd
from PIL import Image

# ===== PATHS =====
SAMPLE_CSV = "/kaggle/input/competitions/dlp-jan-2026-nppe-3/sample_submission.csv"
SUBMISSION_SCRIPT = "/kaggle/input/competitions/dlp-jan-2026-nppe-3/submission.py"
OUTPUT_DIR = "/kaggle/working/output"

# ===== LOAD SAMPLE =====
sample = pd.read_csv(SAMPLE_CSV)

print("Expected images:", len(sample))
print("Generated images:", len(os.listdir(OUTPUT_DIR)))

# ===== CHECK 1: COUNT MATCH =====
assert len(sample) == len(os.listdir(OUTPUT_DIR)),"Number of generated images does not match sample submission!"

# ===== CHECK 2: FILE NAMES MATCH =====
missing_files = []
for image_id in sample['ID']:
    file = image_id + ".jpg"
    if not os.path.exists(os.path.join(OUTPUT_DIR, file)):
        missing_files.append(file)

assert len(missing_files) == 0, \
    f"Missing files: {missing_files[:5]}"

print("All files present!")

# ===== CHECK 3: IMAGE SIZE CONSISTENCY =====
first_img_path = os.path.join(OUTPUT_DIR, sample['ID'][0] + ".jpg")
first_size = Image.open(first_img_path).size

for image_id in sample['ID']:
    path = os.path.join(OUTPUT_DIR, image_id + ".jpg")
    size = Image.open(path).size
    assert size == first_size, f"Size mismatch in {image_id}.jpg"

print(f"All images have consistent size: {first_size}")

# ===== COPY SUBMISSION SCRIPT =====
!cp {SUBMISSION_SCRIPT} submission.py

# ===== GENERATE CSV =====
from submission import generate_submission

generate_submission(
    folder_path=OUTPUT_DIR,
    sample_path=SAMPLE_CSV,
    output_csv="/kaggle/working/submission.csv"
)

print("\nFINAL OUTPUT: /kaggle/working/submission.csv")
print("READY FOR SUBMISSION")

Expected images: 300
Generated images: 300
All files present!
All images have consistent size: (1248, 1248)
✅ Submission saved: /kaggle/working/submission.csv
📊 Shape: (300, 101)

FINAL OUTPUT: /kaggle/working/submission.csv
READY FOR SUBMISSION
